In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import re
from wordcloud import WordCloud
from rouge_score import rouge_scorer
import seaborn as sns
import contractions

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_class_weight

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers

import gensim.downloader as api
from gensim.models import Word2Vec

import itertools

import joblib

# Please uncomment if you want to download
# nltk.download('punkt')
# nltk.download('punkt_tab')      
# nltk.download('wordnet')    
# nltk.download('omw-1.4') 
# nltk.download('averaged_perceptron_tagger_eng') 

In [ ]:
try:
    df = pd.read_csv('filtered_data.csv', nrows=10000)
    df.to_csv('10000only.csv', index=False)
except:
    df = pd.read_csv('10000only.csv')
finally:
    display(df)


# Data Exploration

In [ ]:
df['article_len'] = df['article'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,6))
plt.hist(df['article_len'], bins=50, edgecolor='black' )
plt.title("Article Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df['summary_len'] = df['highlights'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,6))
plt.hist(df['summary_len'], bins=50, edgecolor='black')
plt.title("Summary Length Distribution")
plt.xlabel("Number of Words")
plt.xlim(0, 300)
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

# Normal points
plt.scatter(df['article_len'], df['summary_len'],
            s=10, label='Normal')

# highlight extreme outlier
plt.scatter(df[df['summary_len'] > 400]['article_len'], df[df['summary_len'] > 400]['summary_len'],
            s=20, color='red', label='Outliers')

plt.title("Article Length vs Summary Length")
plt.xlabel("Article Length")
plt.ylabel("Summary Length")
plt.legend()

plt.show()

In [ ]:
# simple cleaning for visualization
def clean_text(text):
    words = text.lower().split()
    words = [w for w in words if w.isalpha()]
    return " ".join(words)

articles_text = " ".join(df['article'].apply(clean_text))
highlights_text = " ".join(df['highlights'].apply(clean_text))

wordcloud_articles = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate(articles_text)

wordcloud_highlights = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='viridis'
).generate(highlights_text)


fig, ax = plt.subplots(1, 2, figsize=(15,6))

ax[0].imshow(wordcloud_articles, interpolation='bilinear')
ax[0].set_title("Articles")
ax[0].axis("off")

ax[1].imshow(wordcloud_highlights, interpolation='bilinear')
ax[1].set_title("Highlights")
ax[1].axis("off")

plt.show()

In [ ]:
# with stopwords
wordcloud_articles = WordCloud(
    width=800,
    height=400,
    background_color='white',
    stopwords=set()
).generate(articles_text)

wordcloud_highlights = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='viridis',
    stopwords=set()
).generate(highlights_text)


fig, ax = plt.subplots(1, 2, figsize=(15,6))

ax[0].imshow(wordcloud_articles, interpolation='bilinear')
ax[0].set_title("Articles")
ax[0].axis("off")

ax[1].imshow(wordcloud_highlights, interpolation='bilinear')
ax[1].set_title("Highlights")
ax[1].axis("off")

plt.show()

# Preprocessing

In [ ]:
df_clean = pd.DataFrame()
df_clean = df.drop(columns=['id','article_len','summary_len']).copy()

In [ ]:
df_clean

In [ ]:
# remove bracketed publisher info
pattern1 = re.compile(
    r'^(?:[a-z,]+\s+){0,3}\((?:[a-z\.]+\s*){1,2}\)(?:\s+--\s+)?',
    re.IGNORECASE
)

# remove bylines, social media follows , publication dates, and update timestamps
pattern2 = re.compile(
    # r'^(?:By\s+.*?[a-z,\.@ ]+?\s+\.\s+)?(?:follow.*?\s*\.\s*)?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s*\.\s*)?',
    # r'^(?:By\s+.*?(?:[\w,\.@]+?\s+\.\s+){0,3})?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s*\.\s*)?',
    r'^(?:By\s+.*?(?:(?:[\w,\.@\s\[\]\']+){0,3}\s+\.\s+){0,3})?(?:PUBLISHED:.*?\|)?(?:.*?UPDATED:\s\..*?\d{1,2}:\d{2}\s+[a-z\.]{2,5},\s*\d{1,2}\s+\w+\s+\d{4}\s+\.\s+)?',
    re.IGNORECASE
)

# remove "Last updated" lines
pattern3 = re.compile(
    r'^Last updated.*?\s*\.\s*',
    re.IGNORECASE
)

### Function Definitions

In [ ]:
def remove_prefix(text):
    text = pattern1.sub('', text)
    text = pattern2.sub('', text)
    text = pattern3.sub('', text)
    return text

In [ ]:
# remove new lines \n
def organize_highlights(text):
    sentences = sent_tokenize(text)
    return " ".join(sentences)

### Apply functions on dataframe

In [ ]:
df_clean["article"] = df_clean["article"].apply(remove_prefix)
df_clean["highlights"] = df_clean["highlights"].apply(organize_highlights)
df_clean

In [ ]:
df_clean.to_csv('cleaned_data.csv', index=False)

In [ ]:
lemmatizer= WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [ ]:
def preprocess(text):
    tokens = word_tokenize(contractions.fix(text.lower()))
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words and len(word)>1
    ]
    return tokens

# Load cleaned data directly to save time

In [ ]:
df_clean = pd.read_csv('cleaned_data.csv')

# Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_clean["article"], df_clean["highlights"], test_size=0.2, random_state=42)

# Logistic Regression

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True) # use stemmer on top of lemmatization for better matching    

In [ ]:
def prepare_data(x, y, threshold=0.15):
    sentences = []
    labels = []

    for article, highlight in zip(x,y):

        article_sentences = sent_tokenize(article.lower())
        highlights_tokens =  preprocess(highlight)

        highlights_text = " ".join(highlights_tokens)

        for sentence in article_sentences:

            words = preprocess(sentence)
            sentence_text = " ".join(words)
            if len(sentence_text)<1:
                break

            # ROUGE similarity score
            score = scorer.score(sentence_text, highlights_text)['rouge1'].fmeasure

            label = 1 if score > threshold else 0

            sentences.append(sentence_text)
            labels.append(label)

    return sentences, labels

In [ ]:
sentences_train, labels_train = prepare_data(X_train, y_train)
sentences_test, labels_test = prepare_data(X_test, y_test)

In [ ]:
sentences_all = sentences_train + sentences_test

In [ ]:
sentences_all

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(sentences_all)

In [ ]:
X_train_v = vectorizer.transform(sentences_train)
X_test_v = vectorizer.transform(sentences_test)

In [ ]:
lr = LogisticRegression()
lr.fit(X_train_v, labels_train)

# ML Evaluation

In [ ]:
preds = lr.predict(X_test_v)
df_ml_comp = pd.DataFrame({'Actual': labels_test[:10], 'Predicted': preds[:10]})
df_ml_comp

In [ ]:
print("Logistic Regression Accuracy:", accuracy_score(labels_test, preds))
print("\nClassification Report:\n", classification_report(labels_test, preds))

In [ ]:
f1_score(labels_test, preds)

In [ ]:
labels_all = labels_train+labels_test
pd.Series(labels_all).value_counts()

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x=labels_all)
plt.title("Label Distribution")
plt.xlabel("Label")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Accuracys is not a good metric for this task due to class imbalance and the nature of summarization.

### Baseline Model

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train_v, labels_train)
preds_baseline = baseline.predict(X_test_v)

In [ ]:
print("Baseline Accuracy:", accuracy_score(labels_test, preds_baseline))
print("\nClassification Report:\n", classification_report(labels_test, preds_baseline, zero_division=0))

### Tuned Model

In [ ]:
logreg = LogisticRegression(max_iter=1000)

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10],                     # Regularization strength, smaller values specify stronger regularization (may underfit). vice versa for larger values (may overfit)
    'penalty': ['l2'],                     # Penalty type, l2 is standard and more stable with sparse data.
    'solver': ['lbfgs', 'liblinear'],      # solvers that support l2
    'class_weight': [None, 'balanced']     # important for imbalanced data
}

# Grid search
grid = GridSearchCV(
    estimator=logreg,
    param_grid=param_grid,
    scoring='f1',     # accuracy is not suitable
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Train
grid.fit(X_train_v, labels_train)

# Best results
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

In [ ]:
lr_best = grid.best_estimator_
preds_best = lr_best.predict(X_test_v)

In [ ]:
cm_baseline = confusion_matrix(labels_test, preds_baseline)
cm_lr = confusion_matrix(labels_test, preds)
cm_lr_best = confusion_matrix(labels_test, preds_best)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title("Baseline")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title("LogReg")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

sns.heatmap(cm_lr_best, annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_title("LogReg Best")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
def summarize_article(article, model, vectorizer, max_words=50, max_sentences=3):

    source_sentences = sent_tokenize(article)

    pre_sentences = [" ".join(preprocess(sentence)) for sentence in source_sentences]
    sentence_vectors = vectorizer.transform(pre_sentences)

    scores = model.predict_proba(sentence_vectors)[:, 1]

    ranked_indices = np.argsort(scores)[::-1]

    selected_sentences = []
    total_words = 0

    for i in ranked_indices:
        sentence = source_sentences[i]
        word_count = len(sentence.split())
        selected_sentences.append((i, sentence))
        total_words += word_count

        # stop if we exceeded limits
        if (total_words > max_words) or (len(selected_sentences) >= max_sentences):
            break

    # sort sentences to be in original article order instead of by article importance
    # improves readability
    selected_sentences = sorted(selected_sentences, key=lambda x: x[0])

    summary = " ".join([s for _, s in selected_sentences])

    return summary

In [ ]:
def get_rouge_score(model, test_data_x, test_data_y, vectorizer, max_words=50, max_sentences=3):
    scores = []
    for article, highlight in zip(test_data_x,test_data_y):
        generated_summary = summarize_article(article, model, vectorizer, max_words,max_sentences)

        score = scorer.score(generated_summary, highlight)['rouge1'].fmeasure
        scores.append(score)
        
    return scores

In [ ]:
def find_rouge_outliers(scores):
    scores = np.array(scores)

    q1 = np.percentile(scores, 25)
    q3 = np.percentile(scores, 75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_indices = np.where((scores < lower_bound) | (scores > upper_bound))[0]

    return {
        "outlier_indices": outlier_indices,
        "outlier_scores": scores[outlier_indices],
        "bounds": (lower_bound, upper_bound)
    }

In [ ]:
rouge_scores = get_rouge_score(lr, X_test, y_test, vectorizer)
len(rouge_scores)

In [ ]:
np.median(rouge_scores)

In [ ]:
find_rouge_outliers(rouge_scores)

In [ ]:
# Since outliers are present use median for ROUGE score representation
df_metrics = pd.DataFrame({
    "Baseline": [
        accuracy_score(labels_test, preds_baseline),
        precision_score(labels_test, preds_baseline, zero_division=0),
        recall_score(labels_test, preds_baseline, zero_division=0),
        f1_score(labels_test, preds_baseline, zero_division=0),
        np.median(get_rouge_score(baseline, X_test, y_test, vectorizer))
    ],
    "LogReg": [
        accuracy_score(labels_test, preds),
        precision_score(labels_test, preds, zero_division=0),
        recall_score(labels_test, preds, zero_division=0),
        f1_score(labels_test, preds, zero_division=0),
        np.median(get_rouge_score(lr, X_test, y_test, vectorizer))
    ],
    "LogReg Best": [
        accuracy_score(labels_test, preds_best),
        precision_score(labels_test, preds_best, zero_division=0),
        recall_score(labels_test, preds_best, zero_division=0),
        f1_score(labels_test, preds_best, zero_division=0),
        np.median(get_rouge_score(lr_best, X_test, y_test, vectorizer))
    ]
}, index=["Accuracy", "Precision", "Recall", "F1", "ROUGE-1 F1"])

display(df_metrics)

# ML extractive summaries (uncomment inputs)

In [ ]:
article = 0
summary_sents = 3
max_words = 50
# article = int(input("Enter the article index (0-9999): "))
# summary_sents = int(input("Enter the number of sentences for the summary: "))
# max_words = int(input("Enter the number of words for the summary: "))
test_article = df_clean['article'].iloc[article]
actual_highlight = df_clean['highlights'].iloc[article]

generated_summary = summarize_article(test_article, lr_best, vectorizer, max_words,summary_sents)

print("\n------- Original Article Snippet ---")
print(test_article[:300] + "...")
print("\n--- Actual Highlight -------")
print(actual_highlight)
print("\n--- Logistic Regression Predicted Summary ---")
print(generated_summary)

# Deep Learning

In [ ]:
sentences_list = [[sent] for sent in sentences_all]
emb_dim = 300

# train Word2Vec
w2v = Word2Vec(
    sentences=sentences_list,
    vector_size=emb_dim,    
    window=5,               # context size
    min_count=2,            # ignore rare words
    workers=5               # CPU cores
)

In [ ]:
# use word2vec average for sentence embedding
def sentence_vector(sentence, model, dim=emb_dim):
    words = preprocess(sentence)
    
    vectors = [model.wv[w] for w in words if w in model.wv]
    
    if len(vectors) == 0:
        return np.zeros(dim)
    
    return np.mean(vectors, axis=0)

In [ ]:
X_dl = []
y_dl = []
    
for sent, label in zip(sentences_all, labels_all):
    X_dl.append(sentence_vector(sent, w2v, emb_dim))
    y_dl.append(label)

X_dl = np.array(X_dl)
y_dl = np.array(y_dl)

In [ ]:
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_dl, y_dl, test_size=0.2, random_state=42)

In [ ]:
np.shape(y_train_dl)

In [ ]:
model_base = keras.Sequential([
    layers.Input(shape=(emb_dim,)),
    
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(1, activation='sigmoid') # probabilties between 0 and 1
])

In [ ]:
model_base.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=["accuracy"]
)

In [ ]:
model_base.fit(
    X_train_dl, y_train_dl,
    validation_data=(X_test_dl, y_test_dl),
    epochs=5,
    batch_size=64
)

In [ ]:
def summarize_article_dl(article, model, max_words=50, max_sentences=3):
    source_sentences = sent_tokenize(article)

    pre_sentences = [" ".join(preprocess(sentence)) for sentence in source_sentences]

    sentence_vectors = np.array([
        sentence_vector(s, w2v, emb_dim)
        for s in pre_sentences
    ])
    
    scores = model.predict(sentence_vectors, verbose=0).flatten()
    
    ranked_indices = np.argsort(scores)[::-1]
    
    selected_sentences = []
    total_words = 0

    for i in ranked_indices:
        sentence = source_sentences[i]
        word_count = len(sentence.split())
        selected_sentences.append((i, sentence))
        total_words += word_count

        # stop if we exceeded limits
        if (total_words > max_words) or (len(selected_sentences) >= max_sentences):
            break

    # sort sentences to be in original article order instead of by article importance
    # improves readability
    selected_sentences = sorted(selected_sentences, key=lambda x: x[0])
    
    summary = " ".join([s for _, s in selected_sentences])
    return summary

In [ ]:
def get_rouge_score_dl(model, test_data_x, test_data_y, max_words=50, max_sentences=3):
    scores = []
    for article, highlight in zip(test_data_x,test_data_y):
        generated_summary = summarize_article_dl(article, model, max_words,max_sentences)

        score = scorer.score(generated_summary, highlight)['rouge1'].fmeasure
        scores.append(score)
        
    return scores

# Deep learning hyperparameter tuning

In [ ]:
# define class weights to aid with imbalance
classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_dl
)

class_weight = {
    0: weights[0],
    1: weights[1]
}

print(class_weight)

In [ ]:
param_grid = {
    "dense1": [64, 128],
    "dense2": [32, 64],
    "dropout": [0.1, 0.2],
    "lr": [1e-3, 1e-4],
    "batch_size": [32, 64]
}

In [ ]:
def build_model(dense1, dense2, dropout, lr):
    model = keras.Sequential([
        layers.Input(shape=(emb_dim,)),
        
        layers.Dense(dense1, activation='relu'),
        layers.Dropout(dropout),
        
        layers.Dense(dense2, activation='relu'),
        layers.Dropout(dropout),
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [ ]:
import itertools

best_score = -1
best_params = None
best_model = None

for dense1, dense2, dropout, lr, batch_size in itertools.product(
    param_grid["dense1"],
    param_grid["dense2"],
    param_grid["dropout"],
    param_grid["lr"],
    param_grid["batch_size"]
):
    
    print(f"Training: {dense1}, {dense2}, {dropout}, {lr}, {batch_size}")
    
    model = build_model(dense1, dense2, dropout, lr)
    
    history = model.fit(
        X_train_dl, y_train_dl,
        validation_data=(X_test_dl, y_test_dl),
        epochs=3,          
        batch_size=batch_size,
        class_weight=class_weight,
        verbose=0
    )
    
    val_acc = history.history['val_accuracy'][-1]
    
    if val_acc > best_score:
        best_score = val_acc
        best_params = {"dense1":dense1, "dense2":dense2, "dropout":dropout, "lr":lr, "batch_size":batch_size}
        best_model = model

print("Best Params: ",best_params)

In [ ]:
model_best = build_model(best_params["dense1"], best_params["dense2"], best_params["dropout"], best_params["lr"])

In [ ]:
model_best.fit(
    X_train_dl, y_train_dl,
    validation_data=(X_test_dl, y_test_dl),
    epochs=5,
    batch_size=best_params["batch_size"]
)

In [ ]:
preds_dl_base = model_base.predict(X_test_dl)
preds_dl_base = (preds_dl_base > 0.3).astype(int)

In [ ]:
preds_dl_best = model_best.predict(X_test_dl)
preds_dl_best = (preds_dl_best > 0.3).astype(int)   # probs greater than 0.3 are treated as 1 (should be generous because most sentences get very low probability)

# Deep learning models evaluation

In [ ]:
print("Classification report DL baseline:\n", classification_report(y_test_dl, preds_dl_base, zero_division=0))

In [ ]:
print("Classification report DL best:\n", classification_report(y_test_dl, preds_dl_best, zero_division=0))

In [ ]:
cm_dl_baseline = confusion_matrix(y_test_dl, preds_dl_best)
cm_dl_best = confusion_matrix(y_test_dl, preds_dl_best)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(cm_dl_baseline, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title("Baseline DL")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_dl_best, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title("Best DL")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

# Deep learning Model extractive summaries

In [ ]:
article = 786
summary_sents = 3
max_words = 50
# article = int(input("Enter the article index (0-9999): "))
# summary_sents = int(input("Enter the number of sentences for the summary: "))
# max_words = int(input("Enter the number of words for the summary: "))
test_article = df_clean['article'].iloc[article]
actual_highlight = df_clean['highlights'].iloc[article]

generated_summary = summarize_article_dl(test_article, model_best, max_words,summary_sents)

print("\n------- Original Article Snippet ---")
print(test_article[:300] + "...")
print("\n--- Actual Highlight -------")
print(actual_highlight)
print("\n--- Logistic Regression Predicted Summary ---")
print(generated_summary)

# Final Models comparison

In [ ]:
df_metrics["DL Base"] = [
    accuracy_score(y_test_dl, preds_dl_base),
    precision_score(y_test_dl, preds_dl_base),
    recall_score(y_test_dl, preds_dl_base),
    f1_score(y_test_dl, preds_dl_base),
    np.median(get_rouge_score_dl(model_base, X_test, y_test)),
]
df_metrics["DL Best"] = [
    accuracy_score(y_test_dl, preds_dl_best),
    precision_score(y_test_dl, preds_dl_best),
    recall_score(y_test_dl, preds_dl_best),
    f1_score(y_test_dl, preds_dl_best),
    np.median(get_rouge_score_dl(model_best, X_test, y_test)),
]

In [ ]:
df_metrics

# Save best Model

In [ ]:
joblib.dump(lr_best, "lr_best_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")